In [ ]:
#| hide
from vruksha import *

# vruksha

> a typed entity graph over a litesearch chunk store, built by an LLM you bring

vruksha turns a litesearch store into a typed knowledge graph. It extracts entities and typed
relations from each chunk with a model you pass in, primes the model with the entities already in
the graph so it reuses them, merges the duplicates that slip through behind a lexical guard, and
searches by walking `refers_to`/`cites`/`builds_on` edges fused with hybrid.

It is model-agnostic: you inject `chat` (any object with `.oneshot(prompt, sp=, max_tokens=)`) and
`emb_fn`. rishi, urai, an API client, anything. vruksha names no model and owns litesearch's
entity/mention/edge tables.

In [ ]:
#| eval: false
from litesearch import database
from vruksha import build_graph          # extract + prime + write + embed + merge, incremental

db = database('corpus.db')               # a litesearch store with a tree
build_graph(db, chat=my_chat, emb_fn=my_encoder)   # a local model does the extraction
db.graph_search('how does resolution work', qemb)  # hybrid + a typed-edge walk

## The lexical guard

Priming stops most duplicates; `resolve_entities` merges the rest. Embedding similarity alone
merges `python 3.11` into `python 3.12`, so every merge passes `_lex_ok` first: token overlap or a
matching acronym, and identical numbers.

In [ ]:
from vruksha.entities import _lex_ok
_lex_ok('multi-head attention', 'multi head attention'), _lex_ok('python 3.11', 'python 3.12')

## A per-corpus citation seed

The model catches loose references. Structured ones (an EU directive's `Article 5`, `Annex III`)
are cheaper caught by a regex, so a legal corpus can pass `seed_fn=cites, canon=norm_cite`. It is
optional and off by default: nothing but a citation-dense corpus needs it.

## What it is worth

On cross-reference bridges (the query shares no word with the answer), the typed leg reaches the
target 0.61 to 0.87 of the time against 0.08 to 0.20 for hybrid, tracking the extractor: a local
Qwen3-4B matches a frontier model, a 1.5B needs the seed. The PMI co-occurrence graph this package
used to build reached none of them; the method and numbers are in litesearch `evals/RESULTS.md`.

## Install

``` sh
pip install vruksha
```